# OpenNext CMS — FLAN-T5 Layout Planner (Stage 1) — Colab Fine-Tuning

Fine-tunes **google/flan-t5-base** (~250M params) to map a natural-language website brief →
**structure-only** `LayoutPlan` JSON (no marketing copy). Copy is filled downstream by the
existing OpenRouter/Gemini Stage-2 filler — this model is **not** trained on copy.

**Runtime:** Colab GPU (Runtime ▸ Change runtime type ▸ **T4 GPU**). ~8 GB VRAM is plenty for
`flan-t5-base` at `batch_size=4`, `max_target_length=1024`, fp16. A T4 (16 GB) trains the
~500-sample set in a few minutes/epoch.

**Inputs you upload:** `train.jsonl` and `val.jsonl` from
`training/data/seq2seq/` (built locally by `build_seq2seq.py`). Each line is
`{"input_text": "plan layout: <brief>", "target_text": "<minified LayoutPlan JSON>"}`.

**Output:** `layout-planner-flan-t5-base/` containing `model.safetensors` (~990 MB) + tokenizer,
which you download (or copy to Drive) and place at
`training/layout-planner/outputs/layout-planner-flan-t5-base/` locally for `inference.py`.

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" sentencepiece safetensors
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only (training will be slow)')

## 2. Provide the data

Two options — run **either** cell.

**Option A — upload** `train.jsonl` + `val.jsonl` directly (simplest).
**Option B — Google Drive** if you keep the files there.

In [ ]:
# Option A: upload train.jsonl and val.jsonl from your machine
from google.colab import files
import os
print('Select train.jsonl and val.jsonl ...')
uploaded = files.upload()
DATA_DIR = '/content'
assert os.path.exists(f'{DATA_DIR}/train.jsonl'), 'train.jsonl not uploaded'
assert os.path.exists(f'{DATA_DIR}/val.jsonl'), 'val.jsonl not uploaded'
print('OK:', os.listdir(DATA_DIR))

In [ ]:
# Option B (alternative): mount Drive instead of uploading
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/opennext/seq2seq'   # <-- adjust to where you put the JSONL
# import os; print(os.listdir(DATA_DIR))

## 3. Config

In [ ]:
MODEL_NAME        = 'google/flan-t5-base'
OUT_DIR           = 'layout-planner-flan-t5-base'
MAX_SOURCE_LENGTH = 128     # 'plan layout: <brief>' is short
MAX_TARGET_LENGTH = 1024    # a full minified LayoutPlan JSON
NUM_EPOCHS        = 4       # 3-5 is the sweet spot for this small set
BATCH_SIZE        = 4       # falls back to 1 + grad-accum on CUDA OOM
LEARNING_RATE     = 3e-4
SEED              = 42

## 4. Load tokenizer + model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print('params (M):', round(model.num_parameters()/1e6, 1))

## 5. Load + tokenize the dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files={'train': f'{DATA_DIR}/train.jsonl',
                                      'validation': f'{DATA_DIR}/val.jsonl'})
print(ds)
print('example input :', ds['train'][0]['input_text'])
print('example target:', ds['train'][0]['target_text'][:160], '...')

In [ ]:
def preprocess(batch):
    model_inputs = tokenizer(batch['input_text'], max_length=MAX_SOURCE_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch['target_text'], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)

# How many targets are being truncated at MAX_TARGET_LENGTH? (want ~0%)
import numpy as np
tl = [len(x) for x in tokenized['train']['labels']]
print(f'target tokens: max={max(tl)} mean={np.mean(tl):.0f} p95={np.percentile(tl,95):.0f}')
print(f'truncated at {MAX_TARGET_LENGTH}: {sum(t>=MAX_TARGET_LENGTH for t in tl)}/{len(tl)}')

## 6. Train

Tries `per_device_train_batch_size=4`; on a CUDA out-of-memory error it automatically
retries with batch size **1** and gradient accumulation 4 (same effective batch).

In [ ]:
from transformers import (Seq2SeqTrainingArguments, Seq2SeqTrainer,
                          DataCollatorForSeq2Seq)
import transformers, inspect

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding='longest')

def make_trainer(bs, grad_accum):
    # eval/save arg name changed across transformers versions; detect it.
    strat_key = 'eval_strategy' if 'eval_strategy' in inspect.signature(
        Seq2SeqTrainingArguments.__init__).parameters else 'evaluation_strategy'
    args_kwargs = dict(
        output_dir=f'./{OUT_DIR}',
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        save_strategy='epoch',
        logging_steps=25,
        load_best_model_at_end=True,
        fp16=torch.cuda.is_available(),
        report_to='none',
        seed=SEED,
    )
    args_kwargs[strat_key] = 'epoch'
    args = Seq2SeqTrainingArguments(**args_kwargs)
    return Seq2SeqTrainer(model=model, args=args, train_dataset=tokenized['train'],
                          eval_dataset=tokenized['validation'], data_collator=data_collator,
                          tokenizer=tokenizer)

try:
    trainer = make_trainer(BATCH_SIZE, 1)
    trainer.train()
except RuntimeError as e:
    if 'out of memory' not in str(e).lower():
        raise
    print('CUDA OOM at batch', BATCH_SIZE, '-> retrying with batch 1 + grad-accum 4')
    torch.cuda.empty_cache()
    trainer = make_trainer(1, 4)
    trainer.train()

## 7. Save weights (safetensors) + tokenizer

In [ ]:
import os
trainer.save_model(OUT_DIR)          # writes model.safetensors by default
tokenizer.save_pretrained(OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))
assert os.path.exists(f'{OUT_DIR}/model.safetensors'), 'model.safetensors not written!'
mb = os.path.getsize(f'{OUT_DIR}/model.safetensors')/1e6
print(f'model.safetensors: {mb:.0f} MB')

## 8. Smoke test — generate one LayoutPlan

In [ ]:
import json
def plan_layout(brief, max_new_tokens=1024):
    prompt = f'plan layout: {brief.strip()}'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

raw = plan_layout('Create a premium dental clinic website in Ahmedabad.')
print(raw[:600])
try:
    obj = json.loads(raw)
    print('\nparsed OK. blocks:', [b.get('type') for b in obj['page']['blocks']])
except Exception as e:
    print('\n(not valid JSON yet — expected on tiny/underfit data):', e)

## 9. Export the model

Download the zip, or copy the folder to Drive. Then extract locally into
`training/layout-planner/outputs/layout-planner-flan-t5-base/`.

In [ ]:
# Option A: zip + download
import shutil
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
from google.colab import files
files.download(f'{OUT_DIR}.zip')

In [ ]:
# Option B: copy to Drive (mount first)
# from google.colab import drive; drive.mount('/content/drive')
# import shutil, os
# dest = '/content/drive/MyDrive/opennext/layout-planner-flan-t5-base'
# os.makedirs(os.path.dirname(dest), exist_ok=True)
# shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
# print('copied to', dest)